# IR (HIT-UAV) - 02 Entrenar

Que hace: entrena un YOLO sobre HIT-UAV (termico) y guarda el run (pesos, metricas, plots) directo en Drive.

Que NO hace: no evalua en test ni procesa video (eso es `03_testing`).

## Parametros

Prueba barata antes de un entrenamiento largo: `EPOCHS = 1` y `TRAIN_ARGS["fraction"] = 0.05`.
Si Colab se corto a mitad: poner en `RESUME_RUN` el nombre de la carpeta del run y correr todo de nuevo.

In [ ]:
DATASET_ZIP = "MyDrive/deteccion_drones/datasets/hituav.zip"
DATASET_DIR = "hituav"                          # subcarpeta en env.DATA_ROOT
DATA_YAML   = "hit-uav/dataset.yaml"            # relativo a DATASET_DIR
OUT_DIR     = "MyDrive/deteccion_drones/runs/ir"

MODEL = "yolov8s.pt"          # o "yolo11s.pt", "yolov8n.pt", o ruta a un .pt propio
EPOCHS, IMGSZ, BATCH = 100, 640, -1   # 640 = resolucion nativa (640x512). BATCH=-1: AutoBatch segun la GPU
TAG = "base"                  # texto libre para el nombre del run: que cambia respecto al anterior
TRAIN_ARGS = dict(
    classes=[0, 1, 2, 3],     # sin DontCare (indice 4): marca zonas ambiguas, no objetos
    hsv_h=0.0, hsv_s=0.0,     # imagen gris: no inventar tono/saturacion que la camara termica no produce
    patience=20, workers=2,
)   # van tal cual a YOLO.train

RESUME_RUN = None             # nombre de un run cortado para retomarlo, ej "hit-uav_yolov8s_640_base_20260928-1530"

## Setup

Identica en todos los notebooks. El repo tiene que ser publico y `BRANCH` tiene que estar pusheada.

In [ ]:
REPO = "https://github.com/sbstn-sss/deteccion_drones.git"
REPO_DIR = "/content/deteccion_drones"
BRANCH = "feat/framework"   # cambiar para probar una rama antes del merge, ej "feat/framework"

import sys
if "google.colab" in sys.modules:
    !git -C {REPO_DIR} checkout -q {BRANCH} 2>/dev/null && git -C {REPO_DIR} pull -q 2>/dev/null || git clone -q -b {BRANCH} {REPO} {REPO_DIR}
    !pip install -q -e {REPO_DIR}
    sys.path.insert(0, f"{REPO_DIR}/src")   # pip -e usa un .pth que el kernel ya iniciado no lee: sin esto falla el import

try:  # el IPython de Colab con Python 3.13 trae un autoreload que importa 'imp' (eliminado en 3.12)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload no disponible: reinicia el kernel tras editar src/")
import importlib; importlib.invalidate_caches()   # el repo se clono despues de arrancar el kernel
from deteccion import env, data, viz
from deteccion import ExperimentConfig, Experiment, list_runs
env.mount_drive()

## 1. Obtener el dataset

Igual que en `01_explorar`: si el dataset ya esta descomprimido en esta sesion (marcador `.done`), no hace nada.

In [ ]:
dataset_dir = env.DATA_ROOT / DATASET_DIR
env.unzip_once(env.drive_path(DATASET_ZIP), dataset_dir)
data_yaml = env.fix_data_yaml(dataset_dir / DATA_YAML)

## 2. Configurar el experimento

El nombre del run se arma solo (dataset, modelo, imgsz, tag, fecha): no hay que adivinar sufijos `-2`, `-8`.
La config queda guardada en `experiment.json` dentro del run.

In [ ]:
cfg = ExperimentConfig(data_yaml=str(data_yaml), out_dir=OUT_DIR, model=MODEL, epochs=EPOCHS,
                       imgsz=IMGSZ, batch=BATCH, tag=TAG, train_args=TRAIN_ARGS)
if RESUME_RUN is None:
    exp = Experiment(cfg)
else:
    exp = Experiment.load(env.drive_path(OUT_DIR) / RESUME_RUN, data_yaml=str(data_yaml))
print(exp.run_dir)

## 3. Entrenar

Se escribe en Drive en cada epoca, asi que una desconexion no pierde el run: se retoma con `RESUME_RUN`.

In [ ]:
results = exp.train() if RESUME_RUN is None else exp.resume()

## 4. Curvas y validacion

Los plots son los que Ultralytics ya genero en el run. Las metricas se miran en **val**; test se usa una sola vez al final, en `03_testing`.

In [ ]:
viz.show_run_plots(exp.run_dir)

In [ ]:
exp.val("val")

## Notas

Siguientes experimentos (cambiar UNA cosa por run, anotarla en `TAG`):
1. `base`: 640, yolov8s (este)
2. `hires`: `IMGSZ = 1024` (99.7% de las personas son small)
3. `scale03`: `TRAIN_ARGS["scale"] = 0.3` (menos zoom-out que borre personas)

Comparar en `03_testing` el **recall de Person**, no solo el mAP general.

_(completar: que se cambio y que resultado dio)_